In [89]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", 50)

In [90]:
import pandas as pd

df = pd.read_csv("quality_data.csv")
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)
df["input_text"] = df["input_text"].str.lower()
df["label"] = df["category"] + "_" + df["subcategory"]
df = df[["input_text", "label", "narration", "mode", "type", "category", "subcategory"]]
df.head()

,input_text,label,narration,mode,type,category,subcategory
0,narration: interest from fixed deposit | mode:...,investment_fd_interest,interest from fixed deposit,NEFT,Credit,investment,fd_interest
1,narration: fd interest payout | mode: neft | t...,investment_fd_interest,FD interest payout,NEFT,Credit,investment,fd_interest
2,narration: fixed deposit interest | mode: neft...,investment_fd_interest,Fixed deposit interest,NEFT,Credit,investment,fd_interest
3,narration: bank interest received | mode: neft...,investment_fd_interest,bank interest received,NEFT,Credit,investment,fd_interest
4,narration: fd maturity interest | mode: neft |...,investment_fd_interest,FD maturity interest,NEFT,Credit,investment,fd_interest


In [91]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    df["input_text"],
    df["label"],
    df,
    test_size=0.3,
    random_state=42,
    stratify=df["label"]
)

In [92]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_y_true = []
all_y_pred = []

In [93]:
from gliner import GLiNER

# GLiNER2 model
gliner_model = GLiNER.from_pretrained("urchade/gliner_large-v2")

labels = list(set(y_train))

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\utils\_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [94]:
def gliner_predict(text):
    entities = gliner_model.predict_entities(text, labels)

    if entities:
        best = max(entities, key=lambda x: x["score"])
        return best["label"], best["score"]

    return "unknown_unknown", 0.0

In [102]:
all_y_true = []
all_y_pred = []
all_conf = []   

for train_idx, test_idx in skf.split(df["input_text"], df["label"]):

    X_test_cv = df["input_text"].iloc[test_idx]
    y_test_cv = df["label"].iloc[test_idx]

    labels = list(set(df["label"].iloc[train_idx]))

    for text, actual in zip(X_test_cv, y_test_cv):
        pred, score = gliner_predict(text)   # 👈 capture score

        all_y_true.append(actual)
        all_y_pred.append(pred)
        all_conf.append(score)              # 👈 store it

In [103]:
result_df = df.copy().reset_index(drop=True)

result_df["predicted_label"] = all_y_pred
result_df["confidence"] = all_conf

In [ ]:
print(len(df))
print(len(all_y_pred))
print(len(all_conf))

In [104]:
result_df = df.copy().reset_index(drop=True)
result_df["predicted_label"] = all_y_pred

In [110]:
result_df = df.copy().reset_index(drop=True)

result_df["predicted_label"] = all_y_pred
result_df["confidence"] = all_conf   # 👈 THIS IS MISSING IN YOUR CODE

def confidence_level(c):
    if c >= 0.6:
        return "High"
    elif c >= 0.3:
        return "Medium"
    else:
        return "Low"

result_df["confidence_level"] = result_df["confidence"].apply(confidence_level)

In [111]:
# Actual split
result_df[["category", "subcategory"]] = result_df["label"].str.split("_", n=1, expand=True)

# Predicted split
result_df[["predicted_category", "predicted_subcategory"]] = result_df["predicted_label"].str.split("_", n=1, expand=True)

In [112]:
result_df["correct"] = result_df["label"] == result_df["predicted_label"]

In [113]:
print(result_df.columns)

Index(['input_text', 'label', 'narration', 'mode', 'type', 'category', 'subcategory', 'predicted_label', 'confidence', 'confidence_level', 'predicted_category', 'predicted_subcategory', 'correct'], dtype='object')


In [114]:
result_df = df_test.copy()

# sort back using original index
result_df = result_df.sort_index().reset_index(drop=True)

In [115]:
final_df[[
    "narration",
    "category",
    "subcategory",
    "predicted_category",
    "predicted_subcategory",
    "confidence",
    "correct"
]].head(20)

,narration,category,subcategory,predicted_category,predicted_subcategory,confidence,correct
84,Home loan EMI payment,expense,loan,expense,loan,0.975600,True
49,Shopping payment online,expense,shopping,expense,shopping,0.876181,True
56,Broadband bill payment,expense,bills,expense,bills,0.816511,True
42,Amazon purchase,expense,shopping,expense,shopping,0.957930,True
78,Policy premium debit,expense,insurance,expense,insurance,0.923979,True
66,Fastag recharge,expense,transport,expense,loan,0.729974,False
52,Internet bill payment,expense,bills,expense,bills,0.834773,True
26,Salary credited,income,salary,expense,loan,0.596097,False
79,Insurance renewal payment,expense,insurance,expense,insurance,0.912333,True
41,Online food order,expense,food,expense,shopping,0.845543,False


In [116]:
from sklearn.metrics import classification_report
import numpy as np
report = classification_report(all_y_true, all_y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose().round(3)
# F2 Score
def f2_score(p, r):
    if (p + r) == 0:
        return 0
    return (5 * p * r) / (4 * p + r)
report_df["f2_score"] = report_df.apply(
    lambda row: f2_score(row["precision"], row["recall"]),
    axis=1
)
report_df = report_df.round(3)
print("\nClassification Report:")
print(report_df)


Classification Report:
                        precision  recall  f1-score  support  f2_score
expense_bills               1.000   0.750     0.857    8.000     0.789
expense_food                1.000   0.375     0.545    8.000     0.429
expense_health              0.500   0.143     0.222    7.000     0.167
expense_insurance           0.667   0.750     0.706    8.000     0.732
expense_loan                0.194   0.750     0.308    8.000     0.477
expense_shopping            0.875   0.875     0.875    8.000     0.875
expense_transport           0.900   0.900     0.900   10.000     0.900
income_salary               0.000   0.000     0.000    8.000     0.000
investment_fd_booking       0.750   0.750     0.750    8.000     0.750
investment_fd_interest      0.857   0.600     0.706   10.000     0.638
investment_stocks           0.857   0.750     0.800    8.000     0.769
accuracy                    0.615   0.615     0.615    0.615     0.615
macro avg                   0.691   0.604     0.606  

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [118]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(all_y_true, all_y_pred)

print("\nOverall Metrics:")
print("Accuracy:", round(accuracy, 3))
print("Macro F1:", report_df.loc["macro avg", "f1-score"])
print("Weighted F1:", report_df.loc["weighted avg", "f1-score"])


Overall Metrics:
Accuracy: 0.615
Macro F1: 0.606
Weighted F1: 0.619


In [119]:
filtered_df = report_df.drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")

best = filtered_df["f1-score"].idxmax()
worst = filtered_df["f1-score"].idxmin()

print("\nBest Performing Category:")
print(best, filtered_df.loc[best, "f1-score"])

print("\nWorst Performing Category:")
print(worst, filtered_df.loc[worst, "f1-score"])


Best Performing Category:
expense_transport 0.9

Worst Performing Category:
income_salary 0.0
